# 07. (선택 과제) 실시간 추론 웹 애플리케이션

> **선택/심화 과제입니다.** 배포한 SageMaker 엔드포인트를 호출하는 **웹앱**을 만들어
> **로컬 개발 서버**(Streamlit)로 띄워 봅니다.

## 아키텍처
```
브라우저 ──HTTP──▶ Streamlit 로컬 서버(app.py) ──boto3/SigV4──▶ SageMaker 엔드포인트
```
- 웹앱은 폼 입력을 36개 피처 벡터로 조립해 엔드포인트에 보내고, 3개 클래스 확률을 받아 표시합니다.
- 엔드포인트는 공개되지 않으며 **AWS 자격증명(IAM)** 으로만 호출됩니다.

## 전제
`05_deployment.ipynb` 로 엔드포인트가 떠 있어야 합니다. (아직 `06` 의 정리 셀을 실행하지 않았어야 함)

## 0. 엔드포인트 상태 확인

In [ ]:
import os
import boto3
import sagemaker

sess = sagemaker.Session()
%store -r endpoint_name region

sm = boto3.client("sagemaker", region_name=region)
status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
print(endpoint_name, "->", status)
assert status == "InService", "엔드포인트가 InService 상태가 아닙니다. 05 노트북을 먼저 실행하세요."

## 1. 웹앱 파일 작성

`webapp/` 폴더에 앱과 의존성 파일을 생성합니다. (아래 `app.py` 셀에서 `call_endpoint` 함수를 완성하세요.)

In [ ]:
import os
os.makedirs("../webapp", exist_ok=True)
print("webapp dir:", os.path.abspath("../webapp"))

In [ ]:
%%writefile ../webapp/app.py
import os

import boto3
import numpy as np
import pandas as pd
import streamlit as st

# 엔드포인트/리전은 환경변수로 주입합니다 (아래 실행 명령 참고).
ENDPOINT_NAME = os.environ.get("ENDPOINT_NAME", "student-success-endpoint")
REGION = os.environ.get("AWS_REGION", os.environ.get("AWS_DEFAULT_REGION", "us-east-1"))
CLASSES = ["Dropout", "Enrolled", "Graduate"]

# 학습 때와 "동일한 순서"의 36개 피처. 순서가 어긋나면 예측이 엉뚱해집니다.
FEATURES = [
    "Marital status", "Application mode", "Application order", "Course",
    "Daytime/evening attendance", "Previous qualification", "Previous qualification (grade)",
    "Nacionality", "Mother's qualification", "Father's qualification", "Mother's occupation",
    "Father's occupation", "Admission grade", "Displaced", "Educational special needs",
    "Debtor", "Tuition fees up to date", "Gender", "Scholarship holder", "Age at enrollment",
    "International", "Curricular units 1st sem (credited)", "Curricular units 1st sem (enrolled)",
    "Curricular units 1st sem (evaluations)", "Curricular units 1st sem (approved)",
    "Curricular units 1st sem (grade)", "Curricular units 1st sem (without evaluations)",
    "Curricular units 2nd sem (credited)", "Curricular units 2nd sem (enrolled)",
    "Curricular units 2nd sem (evaluations)", "Curricular units 2nd sem (approved)",
    "Curricular units 2nd sem (grade)", "Curricular units 2nd sem (without evaluations)",
    "Unemployment rate", "Inflation rate", "GDP",
]

# 폼에서 받지 않는 피처를 채울 기본값(데이터셋 중앙값).
DEFAULTS = {
    "Marital status": 1, "Application mode": 17, "Application order": 1, "Course": 9238,
    "Daytime/evening attendance": 1, "Previous qualification": 1, "Previous qualification (grade)": 133.1,
    "Nacionality": 1, "Mother's qualification": 19, "Father's qualification": 19,
    "Mother's occupation": 5, "Father's occupation": 7, "Admission grade": 126.1,
    "Displaced": 1, "Educational special needs": 0, "Debtor": 0, "Tuition fees up to date": 1,
    "Gender": 0, "Scholarship holder": 0, "Age at enrollment": 20, "International": 0,
    "Curricular units 1st sem (credited)": 0, "Curricular units 1st sem (enrolled)": 6,
    "Curricular units 1st sem (evaluations)": 8, "Curricular units 1st sem (approved)": 5,
    "Curricular units 1st sem (grade)": 12.286, "Curricular units 1st sem (without evaluations)": 0,
    "Curricular units 2nd sem (credited)": 0, "Curricular units 2nd sem (enrolled)": 6,
    "Curricular units 2nd sem (evaluations)": 8, "Curricular units 2nd sem (approved)": 5,
    "Curricular units 2nd sem (grade)": 12.2, "Curricular units 2nd sem (without evaluations)": 0,
    "Unemployment rate": 11.1, "Inflation rate": 1.4, "GDP": 0.32,
}

runtime = boto3.client("sagemaker-runtime", region_name=REGION)


def call_endpoint(feature_row):
    """feature_row: 길이 36 피처 리스트(FEATURES 순서). 3개 클래스 확률 리스트를 반환."""
    # TODO: SageMaker 엔드포인트를 호출해 확률 리스트를 반환하세요. (06 노트북의 boto3 방식 응용)
    #  1) feature_row 를 콤마로 join 해서 CSV 한 줄 문자열 만들기
    #  2) runtime.invoke_endpoint(EndpointName=ENDPOINT_NAME, ContentType="text/csv", Body=<csv>)
    #  3) resp["Body"].read().decode() -> "0.1,0.2,0.7" -> [float, ...] 로 변환해 반환
    raise NotImplementedError("call_endpoint 를 구현하세요")


st.set_page_config(page_title="학생 학업 성취 예측", page_icon="🎓")
st.title("🎓 학생 학업 성취 예측")
st.caption(f"SageMaker 실시간 엔드포인트: {ENDPOINT_NAME}  ·  region: {REGION}")
st.write("주요 항목만 입력하세요. 나머지 피처는 데이터셋 중앙값으로 자동 채워집니다.")

c1, c2 = st.columns(2)
with c1:
    age = st.slider("입학 시 나이", 17, 60, int(DEFAULTS["Age at enrollment"]))
    admission = st.slider("입학 성적 (0-200)", 0.0, 200.0, float(DEFAULTS["Admission grade"]))
    prev_grade = st.slider("이전 학력 성적 (0-200)", 0.0, 200.0, float(DEFAULTS["Previous qualification (grade)"]))
    gender = st.selectbox("성별", [("여성", 0), ("남성", 1)], format_func=lambda x: x[0])[1]
with c2:
    s1_appr = st.slider("1학기 이수(합격) 과목 수", 0, 26, int(DEFAULTS["Curricular units 1st sem (approved)"]))
    s1_grade = st.slider("1학기 평균 성적 (0-20)", 0.0, 20.0, float(DEFAULTS["Curricular units 1st sem (grade)"]))
    s2_appr = st.slider("2학기 이수(합격) 과목 수", 0, 26, int(DEFAULTS["Curricular units 2nd sem (approved)"]))
    s2_grade = st.slider("2학기 평균 성적 (0-20)", 0.0, 20.0, float(DEFAULTS["Curricular units 2nd sem (grade)"]))

tuition = 1 if st.checkbox("등록금 납부 최신", value=bool(DEFAULTS["Tuition fees up to date"])) else 0
scholarship = 1 if st.checkbox("장학금 수혜", value=bool(DEFAULTS["Scholarship holder"])) else 0
debtor = 1 if st.checkbox("채무 있음", value=bool(DEFAULTS["Debtor"])) else 0

overrides = {
    "Age at enrollment": age,
    "Admission grade": admission,
    "Previous qualification (grade)": prev_grade,
    "Gender": gender,
    "Curricular units 1st sem (approved)": s1_appr,
    "Curricular units 1st sem (grade)": s1_grade,
    "Curricular units 2nd sem (approved)": s2_appr,
    "Curricular units 2nd sem (grade)": s2_grade,
    "Tuition fees up to date": tuition,
    "Scholarship holder": scholarship,
    "Debtor": debtor,
}

if st.button("예측하기", type="primary"):
    values = dict(DEFAULTS)
    values.update(overrides)
    feature_row = [values[f] for f in FEATURES]
    try:
        proba = call_endpoint(feature_row)
        pred = CLASSES[int(np.argmax(proba))]
        st.success(f"예측 결과:  {pred}")
        st.bar_chart(pd.DataFrame({"확률": proba}, index=CLASSES))
    except Exception as e:
        st.error(f"엔드포인트 호출 실패: {e}")
        st.info("엔드포인트가 실행 중인지, AWS 자격증명/리전이 올바른지 확인하세요.")


In [ ]:
%%writefile ../webapp/requirements.txt
streamlit>=1.30
boto3>=1.34
numpy
pandas

## 2. 로컬 개발 서버 실행

Streamlit 서버는 노트북 셀이 아니라 **터미널**에서 실행합니다 (셀에서 실행하면 커널이 멈춥니다).
아래 셀을 실행해 나오는 명령을 복사하세요.

- **SageMaker Studio**: `File ▸ New ▸ Terminal` 로 터미널을 열고 명령 실행.
  실행 후 접속 URL은 Studio 프록시 경로를 사용합니다 → 노트북 URL의 `/lab/...` 부분을
  `/proxy/8501/` 로 바꿔 접속합니다. (jupyter-server-proxy 필요)
- **내 노트북(로컬 PC)**: `aws configure` 로 자격증명을 설정한 뒤 명령을 실행하고
  브라우저에서 `http://localhost:8501` 접속. (IAM 권한에 `sagemaker:InvokeEndpoint` 필요)

In [ ]:
import os
webapp_dir = os.path.abspath("../webapp")
print("=== 터미널에서 실행할 명령 ===\n")
print(f"cd {webapp_dir}")
print("pip install -r requirements.txt")
print(f"ENDPOINT_NAME={endpoint_name} AWS_REGION={region} streamlit run app.py --server.port 8501")

## 3. 🧹 정리

웹앱 실습까지 끝났다면 엔드포인트를 삭제해 과금을 멈춥니다. (이미 06에서 삭제했다면 건너뜁니다.)

In [ ]:
import boto3
sm = boto3.client("sagemaker", region_name=region)
try:
    sm.delete_endpoint(EndpointName=endpoint_name)
    print("deleted endpoint:", endpoint_name)
except Exception as e:
    print("endpoint 삭제 건너뜀:", e)
try:
    sm.delete_endpoint_config(EndpointConfigName=endpoint_name)
    print("deleted endpoint-config:", endpoint_name)
except Exception as e:
    print("endpoint-config 삭제 건너뜀:", e)

🎉 **수고하셨습니다!** 데이터 전처리부터 학습·평가·튜닝·배포·추론, 그리고 웹앱까지 완성했습니다.

> 남은 리소스가 없는지 SageMaker 콘솔의 **Endpoints / Models / Endpoint configurations** 를
> 확인하고, S3 버킷의 워크샵 데이터도 필요 없으면 삭제하세요.